# N170 Per-Subject Analysis

Secondary confirmatory analysis (prerec.md §5). Requires preprocessed epochs
from all four conditions — run notebook 01_pipeline.ipynb first.

Computes N170 amplitude (mean voltage in 130–200 ms at PO7/Oz/PO8) across
ALL clean face-flash epochs regardless of target status, saves results to
`data/derived/n170-v1/`, and plots the posterior ERP per condition.

In [ ]:
# ---- CONFIGURE HERE -------------------------------------------------------
SUBJECT_ID  = '02'             # bare id, e.g. '01'
DERIVED_DIR = 'data/derived'
# ---------------------------------------------------------------------------

In [ ]:
import sys
import os
from pathlib import Path

# Find repo root by walking up until we find config.yaml.
_here = Path('.').resolve()
repo_root = next(
    (p for p in [_here, _here.parent, _here.parent.parent]
     if (p / 'config.yaml').exists()),
    _here,
)
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))
print(f'Working directory: {Path.cwd()}')

%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 120

from analysis.n170 import (
    run_n170_subject,
    N170_CHANNELS, N170_TMIN_S, N170_TMAX_S,
    CONDITIONS,
)
from analysis.plots import (
    plot_n170_erp_overlay,
    plot_n170_amplitude_bar,
)
print('Imports OK')

## 1. Compute and save N170 results

Loads preprocessed epochs for all four conditions and computes the
pre-registered scalar: mean voltage in 130–200 ms at PO7/Oz/PO8, grand-
averaged over all clean epochs (target + nontarget).

In [ ]:
n170_result = run_n170_subject(
    subject_id=SUBJECT_ID,
    derived_root=DERIVED_DIR,
    save=True,
)

print(f'Subject: sub-{SUBJECT_ID}')
print(f'Channels: {N170_CHANNELS}  |  Window: {N170_TMIN_S*1000:.0f}–{N170_TMAX_S*1000:.0f} ms')
print()
print(f'{"Condition":<20} {"N170 (µV)":>12} {"N epochs":>10}')
print('-' * 45)
for cond in CONDITIONS:
    pc = n170_result['per_condition'][cond]
    print(f'{cond:<20} {pc["n170_amplitude_uv"]:>12.3f} {pc["n_epochs"]:>10}')

saved_path = Path(DERIVED_DIR) / 'n170-v1' / f'sub-{SUBJECT_ID}' / f'sub-{SUBJECT_ID}_n170.json'
print(f'\nSaved to: {saved_path}')

## 2. Posterior ERP by condition

Grand-average waveform at the mean of PO7/Oz/PO8 for each condition.
A robust N170 appears as a negative deflection in the shaded 130–200 ms window.
Noise attenuating the N170 will reduce that negative peak.

In [ ]:
fig = plot_n170_erp_overlay(n170_result)
plt.show()

## 3. Per-channel posterior ERPs (control only)

Plots PO7, Oz, and PO8 individually for the control condition to check
whether all three channels carry the N170 signal.

In [ ]:
import numpy as np

ctrl = n170_result['per_condition']['control']
times_ms = np.array(ctrl['evoked_times_s']) * 1000

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True, layout='constrained')
for ax, ch in zip(axes, N170_CHANNELS):
    wave = ctrl['evoked_per_channel_uv'][ch]
    ax.plot(times_ms, wave, color='#333333', linewidth=1.5)
    ax.axvspan(N170_TMIN_S * 1000, N170_TMAX_S * 1000,
               alpha=0.15, color='gold', label='N170 window')
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_title(f'{ch} — control')
    ax.set_xlabel('Time (ms)')
    ax.set_xlim(times_ms[0], times_ms[-1])
axes[0].set_ylabel('Amplitude (µV)')
fig.suptitle(f'sub-{SUBJECT_ID}  posterior channels (control, all epochs)', fontsize=12)
plt.show()

## 4. N170 amplitude per condition

In [ ]:
fig = plot_n170_amplitude_bar(n170_result)
plt.show()

## 5. Per-channel ERPs across all conditions

Each channel plotted separately, all four conditions overlaid. Useful for
checking whether amplitude differences are consistent across channels.

In [ ]:
from analysis.plots import CONDITION_COLORS

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True, layout='constrained')
for ax, ch in zip(axes, N170_CHANNELS):
    for cond in CONDITIONS:
        pc = n170_result['per_condition'][cond]
        times_ms = np.array(pc['evoked_times_s']) * 1000
        wave = pc['evoked_per_channel_uv'][ch]
        ax.plot(times_ms, wave,
                color=CONDITION_COLORS.get(cond, '#888888'),
                linewidth=1.5, label=cond)
    ax.axvspan(N170_TMIN_S * 1000, N170_TMAX_S * 1000,
               alpha=0.12, color='gold')
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_title(ch)
    ax.set_xlabel('Time (ms)')
    ax.set_xlim(times_ms[0], times_ms[-1])
axes[0].set_ylabel('Amplitude (µV)')
axes[-1].legend(fontsize=8, loc='lower right')
fig.suptitle(f'sub-{SUBJECT_ID}  posterior channels by condition', fontsize=12)
plt.show()